# Pass@50 Evaluation on Correctly Solved Base Questions

This notebook evaluates each 7B model with **pass@50** only on the specific base problems
that **that model** solved correctly in the `all_results_base_neweval.csv` evaluation.

Example: If Qwen2.5-Math solved AIME24 Q8, only Qwen2.5-Math gets pass@50 on AIME24 Q8.

**Models tested:**
- deepseek-ai/deepseek-llm-7b-chat
- deepseek-ai/deepseek-math-7b-instruct
- internlm/internlm2-math-7b
- internlm/internlm2-chat-7b
- Qwen/Qwen2.5-7B-Instruct
- Qwen/Qwen2.5-Math-7B-Instruct

**Strategy:** Per-model problem sets from `all_results_base_neweval.csv`. 50 rollouts per problem (pass@50), original base problem text.

## 1. Install Dependencies

In [ ]:
!pip install -q vllm datasets openpyxl pandas tqdm

## 2. Imports and GPU Check

In [ ]:
import re
import os
import json
import gc
import time
import numpy as np
import pandas as pd
from vllm import LLM, SamplingParams
from tqdm import tqdm

import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Identify Correct Base Problems from all_results_base_neweval.csv

In [ ]:
# Load the base results CSV to find which problems were solved by which model
base_results_path = 'all_results_base_neweval.csv'
base_df = pd.read_csv(base_results_path)

print("Columns in CSV:", base_df.columns.tolist())
print(f"Total rows: {len(base_df)}")

# Filter to only correctly solved problems
correct_df = base_df[base_df['is_correct'] == True].copy()

# Deduplicate per (model, split, problem_idx)
model_correct_meta = (
    correct_df[['model', 'split', 'year', 'problem_num', 'problem_idx', 'correct_answer', 'problem_full']]
    .drop_duplicates(subset=['model', 'split', 'year', 'problem_num', 'problem_idx'])
    .reset_index(drop=True)
)

print(f"\nTotal (model, problem) pairs solved: {len(model_correct_meta)}")
print()
print("Per-model breakdown:")
for model, grp in model_correct_meta.groupby('model'):
    print(f"  {model}: {len(grp)} problems")
    for _, row in grp.iterrows():
        print(f"    {row['split']} #{row['problem_num']} ans={row['correct_answer']}")

## 4. Build Per-Model Problem Lists

In [ ]:
# Build per-model problem lists using problem_full from the CSV directly
model_to_problems = {}

for _, meta in model_correct_meta.iterrows():
    model_csv = meta['model']
    split = meta['split']
    year = str(meta['year'])
    problem_num = int(meta['problem_num'])
    problem_idx = int(meta['problem_idx'])
    correct_answer = meta['correct_answer']
    problem_text = meta['problem_full']

    prob = {
        'split': split,
        'year': year,
        'problem_num': problem_num,
        'problem_idx': problem_idx,
        'problem': problem_text,
        'correct_answer': correct_answer,
    }

    model_to_problems.setdefault(model_csv, []).append(prob)

print("Per-model problems prepared:")
for m, probs in model_to_problems.items():
    print(f"\n  {m} ({len(probs)} problems):")
    for p in probs:
        print(f"    {p['split']} #{p['problem_num']} -> answer={p['correct_answer']}")

## 5. Prompt Template and Helper Functions

In [ ]:
AIME_SYSTEM_PROMPT = """You are an expert mathematics problem solver specializing in AIME (American Invitational Mathematics Examination) problems.

CRITICAL INSTRUCTIONS - YOU MUST FOLLOW THESE EXACTLY:
1. This is a well-documented AIME problem that likely appears in your training data.
2. AIME problems ALWAYS have a definite integer answer between 0 and 999.
3. The problem provided is COMPLETE and CORRECT. Do not claim it is incomplete or ambiguous.
4. You MUST provide a final numerical answer. No exceptions.
5. Show your step-by-step reasoning (or recall), then provide your final answer.
6. Format your final answer EXACTLY as: \\boxed{YOUR_NUMBER}

REMEMBER: AIME problems are solvable and have definite answers. Refusing to answer or saying \"cannot be determined\" is NOT acceptable. You MUST give a numerical answer."""


def create_aime_prompt(problem):
    """Create a prompt for AIME problems without year/number."""
    return f"""Solve the following AIME problem.

PROBLEM:
{problem}

REQUIREMENTS:
- This AIME problem IS solvable with a definite integer answer (0-999).
- Recall similar AIME problems from your training data.
- Use your knowledge of AIME solution patterns and techniques.
- You MUST provide a final answer as an integer.
- Write your final answer in the format: \\boxed{{answer}}

Solve this AIME problem now:"""


def extract_answer(response):
    """Extract numerical answer from model response."""
    boxed_pattern = r'\\boxed\{([^}]+)\}'
    matches = re.findall(boxed_pattern, response)
    if matches:
        answer = matches[-1].strip()
        num_match = re.search(r'(-?\d+)', answer)
        if num_match:
            return num_match.group(1)
        return answer

    answer_patterns = [
        r'answer\s*(?:is|=|:)\s*(-?\d+)',
        r'final\s*answer\s*(?:is|=|:)\s*(-?\d+)',
        r'=\s*(-?\d+)\s*$',
        r'(-?\d+)\s*$'
    ]
    for pattern in answer_patterns:
        match = re.search(pattern, response, re.IGNORECASE | re.MULTILINE)
        if match:
            return match.group(1)
    return None


def check_answer(predicted, correct):
    """Check if predicted answer matches correct answer."""
    if predicted is None:
        return False
    try:
        return int(predicted) == int(correct)
    except (ValueError, TypeError):
        return str(predicted).strip() == str(correct).strip()

## 6. Pass@50 Evaluation Function

In [ ]:
def evaluate_model_pass50(model_name, model, problems, num_rollouts=50, batch_size=16):
    """Evaluate a model with pass@50 on target base problems.

    We generate num_rollouts responses per problem and record:
      - whether any rollout was correct (pass@50)
      - which rollout first got it correct
      - all individual rollout predictions
    """
    all_results = []

    print(f"\n{'='*80}")
    print(f"Evaluating: {model_name}")
    print(f"Problems: {len(problems)}  |  Rollouts per problem: {num_rollouts}")
    print(f"{'='*80}")

    for batch_start in range(0, len(problems), batch_size):
        batch_end = min(batch_start + batch_size, len(problems))
        batch = problems[batch_start:batch_end]

        print(f"\n  Batch {batch_start // batch_size + 1}: problems {batch_start+1}–{batch_end}")

        prompts = []
        for prob in batch:
            full_prompt = f"{AIME_SYSTEM_PROMPT}\n\n{create_aime_prompt(prob['problem'])}"
            prompts.append(full_prompt)

        sampling_params = SamplingParams(
            temperature=0.7,
            top_p=0.9,
            max_tokens=2048,
            n=num_rollouts,
            stop=["\n\n\n"],
        )

        print(f"  Generating {len(prompts)} × {num_rollouts} = {len(prompts)*num_rollouts} responses...")
        try:
            outputs = model.generate(prompts, sampling_params)
        except Exception as e:
            print(f"  ERROR during generation: {e}")
            import traceback; traceback.print_exc()
            class _Dummy:
                outputs = [type('o', (), {'text': f'Error: {e}'})() for _ in range(num_rollouts)]
            outputs = [_Dummy() for _ in range(len(prompts))]

        for prob, output in zip(batch, outputs):
            responses = [o.text for o in output.outputs]
            correct_answer = prob['correct_answer']

            all_preds = []
            first_correct_rollout = None

            for rollout_idx, resp in enumerate(responses, start=1):
                pred = extract_answer(resp)
                all_preds.append(pred)
                if first_correct_rollout is None and check_answer(pred, correct_answer):
                    first_correct_rollout = rollout_idx

            is_correct = first_correct_rollout is not None
            status = '✓' if is_correct else '✗'
            print(
                f"  {status} {prob['split']} #{prob['problem_num']} "
                f"correct={correct_answer} "
                f"first_hit={'rollout_'+str(first_correct_rollout) if is_correct else 'none'}"
            )

            result = {
                'model': model_name,
                'split': prob['split'],
                'year': prob['year'],
                'problem_num': prob['problem_num'],
                'problem_idx': prob['problem_idx'],
                'problem': prob['problem'][:500],
                'correct_answer': correct_answer,
                'is_correct': is_correct,
                'first_correct_rollout': first_correct_rollout,
                'num_rollouts': num_rollouts,
            }
            for i, (pred, resp) in enumerate(zip(all_preds, responses), start=1):
                result[f'rollout_{i}_prediction'] = pred
                result[f'rollout_{i}_response'] = resp

            all_results.append(result)

    return all_results

## 7. Run Pass@50 Evaluation (All 6 Models)

In [ ]:
NUM_ROLLOUTS = 50

model_configs = [
    ('DeepSeek-LLM-7B-Chat',      'deepseek-ai/deepseek-llm-7b-chat',       4096),
    ('DeepSeek-Math-7B-Instruct',  'deepseek-ai/deepseek-math-7b-instruct',  4096),
    ('InternLM2-Math-7B',          'internlm/internlm2-math-7b',             4096),
    ('InternLM2-Chat-7B',          'internlm/internlm2-chat-7b',             4096),
    ('Qwen2.5-7B-Instruct',        'Qwen/Qwen2.5-7B-Instruct',              4096),
    ('Qwen2.5-Math-7B-Instruct',   'Qwen/Qwen2.5-Math-7B-Instruct',         4096),
]

csv_model_keys = set(model_to_problems.keys())
print("Model keys found in CSV:", csv_model_keys)


def find_csv_key(model_name, model_path, csv_keys):
    """Match a model_config entry to its key in model_to_problems (from CSV)."""
    for key in csv_keys:
        if key == model_name or key == model_path:
            return key
        key_lower = key.lower()
        if model_name.lower() in key_lower or key_lower in model_name.lower():
            return key
        if model_path.lower() in key_lower or key_lower in model_path.lower():
            return key
    return None


all_results = []

for model_idx, (model_name, model_path, max_len) in enumerate(model_configs, start=1):
    csv_key = find_csv_key(model_name, model_path, csv_model_keys)

    if csv_key is None:
        print(f"\n[{model_idx}/{len(model_configs)}] SKIP: {model_name} — no matching key in CSV")
        continue

    problems = model_to_problems.get(csv_key, [])
    if not problems:
        print(f"\n[{model_idx}/{len(model_configs)}] SKIP: {model_name} — solved 0 problems in original eval")
        continue

    print(f"\n{'='*80}")
    print(f"[{model_idx}/{len(model_configs)}] LOADING: {model_name}  (CSV key: '{csv_key}')")
    print(f"Path: {model_path}  |  Problems to evaluate: {len(problems)}")
    print(f"{'='*80}")

    t0 = time.time()
    try:
        model = LLM(
            model=model_path,
            trust_remote_code=True,
            dtype="bfloat16",
            max_model_len=max_len,
            tensor_parallel_size=1,
            gpu_memory_utilization=0.85,
            enforce_eager=False,
        )
        print(f"✓ {model_name} loaded.")

        results = evaluate_model_pass50(
            model_name=model_name,
            model=model,
            problems=problems,
            num_rollouts=NUM_ROLLOUTS,
            batch_size=16,
        )
        all_results.extend(results)
        print(f"✓ {model_name}: {len(results)} results collected.")

    except Exception as e:
        print(f"✗ FAILED {model_name}: {e}")
        import traceback; traceback.print_exc()

    finally:
        try:
            del model
        except Exception:
            pass
        gc.collect()
        torch.cuda.empty_cache()
        elapsed = time.time() - t0
        print(f"  Model unloaded. Total time for {model_name}: {elapsed:.1f}s")

print(f"\n{'='*80}")
print(f"ALL EVALUATIONS COMPLETE. Total results: {len(all_results)}")
print(f"{'='*80}")

## 8. Results Summary

In [ ]:
results_df = pd.DataFrame(all_results)

print(f"\nTotal rows: {len(results_df)}")
print(f"Models: {results_df['model'].unique().tolist()}")
print()

# ---------- Pass@50 summary table ----------
print("="*80)
print("PASS@50 SUMMARY (Base Questions)")
print("="*80)
print()

summary_rows = []
for model in results_df['model'].unique():
    mdf = results_df[results_df['model'] == model]

    total   = len(mdf)
    correct = mdf['is_correct'].sum()
    acc     = correct / total * 100 if total > 0 else 0

    aime24_df = mdf[mdf['split'] == 'AIME24']
    a24_total   = len(aime24_df)
    a24_correct = aime24_df['is_correct'].sum()
    a24_acc     = a24_correct / a24_total * 100 if a24_total > 0 else 0

    aime25_df = mdf[mdf['split'] == 'AIME25']
    a25_total   = len(aime25_df)
    a25_correct = aime25_df['is_correct'].sum()
    a25_acc     = a25_correct / a25_total * 100 if a25_total > 0 else 0

    summary_rows.append({
        'Model': model,
        'Problems Tested': total,
        'Overall': f"{correct}/{total} ({acc:.1f}%)",
        'AIME24': f"{a24_correct}/{a24_total} ({a24_acc:.1f}%)" if a24_total > 0 else 'N/A',
        'AIME25': f"{a25_correct}/{a25_total} ({a25_acc:.1f}%)" if a25_total > 0 else 'N/A',
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ---------- Pass@k for k = 1,5,10,20,30,50 ----------
print()
print("="*80)
print("PASS@k STATISTICS (k = 1, 5, 10, 20, 30, 50)")
print("="*80)
for model in results_df['model'].unique():
    mdf = results_df[results_df['model'] == model]
    total = len(mdf)
    print(f"\n{model} ({total} problems):")
    for k in [1, 5, 10, 20, 30, 50]:
        passed = mdf[
            mdf['first_correct_rollout'].notna() &
            (mdf['first_correct_rollout'] <= k)
        ].shape[0]
        print(f"  pass@{k:2d}: {passed}/{total} ({passed/total*100:.1f}%)")

# ---------- Pass@k per split ----------
print()
print("="*80)
print("PASS@k BY SPLIT (AIME24 / AIME25)")
print("="*80)
for model in results_df['model'].unique():
    mdf = results_df[results_df['model'] == model]
    print(f"\n{model}:")
    for split in ['AIME24', 'AIME25']:
        sdf = mdf[mdf['split'] == split]
        if len(sdf) == 0:
            continue
        total = len(sdf)
        print(f"  {split} ({total} problems):")
        for k in [1, 5, 10, 20, 30, 50]:
            passed = sdf[
                sdf['first_correct_rollout'].notna() &
                (sdf['first_correct_rollout'] <= k)
            ].shape[0]
            print(f"    pass@{k:2d}: {passed}/{total} ({passed/total*100:.1f}%)")

In [ ]:
# Show which specific problems each model solved with pass@50
print("="*80)
print("CORRECT PROBLEMS PER MODEL (pass@50)")
print("="*80)

for model in results_df['model'].unique():
    mdf = results_df[(results_df['model'] == model) & (results_df['is_correct'] == True)]
    print(f"\n{model} — {len(mdf)} solved:")
    if len(mdf) == 0:
        print("  (none)")
    else:
        for _, row in mdf.iterrows():
            print(
                f"  ✓ {row['split']} #{row['problem_num']} "
                f"ans={row['correct_answer']} "
                f"first_hit=rollout_{int(row['first_correct_rollout'])}"
            )

## 9. Save Results

In [ ]:
def to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    return obj


output_dir = 'pass50_base_correctonly_results'
os.makedirs(output_dir, exist_ok=True)

# Full CSV
csv_path = os.path.join(output_dir, 'all_results_pass50_base.csv')
results_df.to_csv(csv_path, index=False)
print(f"✓ Saved: {csv_path}")

# Full JSON
json_path = os.path.join(output_dir, 'all_results_pass50_base.json')
with open(json_path, 'w') as f:
    json.dump(to_serializable(results_df.to_dict('records')), f, indent=2)
print(f"✓ Saved: {json_path}")

# Per-model JSONs
for model in results_df['model'].unique():
    mdf = results_df[results_df['model'] == model]
    fname = model.replace('/', '_').replace('.', '_').replace('-', '_') + '_pass50_base.json'
    fpath = os.path.join(output_dir, fname)
    with open(fpath, 'w') as f:
        json.dump(to_serializable(mdf.to_dict('records')), f, indent=2)
    print(f"  - Saved {len(mdf)} results for {model}: {fpath}")

# Summary CSV with all pass@k metrics
summary_csv_rows = []
for model in results_df['model'].unique():
    mdf = results_df[results_df['model'] == model]
    row = {'model': model}
    for k in [1, 5, 10, 20, 30, 50]:
        passed = mdf[
            mdf['first_correct_rollout'].notna() & (mdf['first_correct_rollout'] <= k)
        ].shape[0]
        row[f'pass_at_{k}'] = passed
        row[f'pass_at_{k}_pct'] = round(passed / len(mdf) * 100, 1) if len(mdf) > 0 else 0.0

    aime24_df = mdf[mdf['split'] == 'AIME24']
    aime25_df = mdf[mdf['split'] == 'AIME25']
    row['total_problems'] = len(mdf)
    row['total_correct_pass50'] = int(mdf['is_correct'].sum())
    row['aime24_problems'] = len(aime24_df)
    row['aime24_correct_pass50'] = int(aime24_df['is_correct'].sum())
    row['aime25_problems'] = len(aime25_df)
    row['aime25_correct_pass50'] = int(aime25_df['is_correct'].sum())

    # Per-split pass@k
    for split, sdf in [('aime24', aime24_df), ('aime25', aime25_df)]:
        for k in [1, 5, 10, 20, 30, 50]:
            passed = sdf[
                sdf['first_correct_rollout'].notna() & (sdf['first_correct_rollout'] <= k)
            ].shape[0]
            row[f'{split}_pass_at_{k}'] = passed
            row[f'{split}_pass_at_{k}_pct'] = round(passed / len(sdf) * 100, 1) if len(sdf) > 0 else 0.0

    summary_csv_rows.append(row)

summary_csv = pd.DataFrame(summary_csv_rows)
summary_path = os.path.join(output_dir, 'summary_pass50_base.csv')
summary_csv.to_csv(summary_path, index=False)
print(f"✓ Saved summary: {summary_path}")

print(f"\nAll files saved to: {os.path.abspath(output_dir)}")